# Deploying Step-3.7-Flash with TensorRT-LLM

This notebook walks you through deploying the `stepfun-ai/Step-3.7-Flash` model (text generation path) using TensorRT-LLM.

[TensorRT-LLM](https://nvidia.github.io/TensorRT-LLM/) is NVIDIA's open-source library for accelerating and optimizing LLM inference on NVIDIA GPUs. Support for Step-3.7-Flash is enabled through the AutoDeploy workflow. More details about AutoDeploy can be found [here](https://nvidia.github.io/TensorRT-LLM/torch/auto_deploy/auto-deploy.html).

**Model Resources:**
- [HuggingFace Model Card](https://huggingface.co/stepfun-ai/Step-3.7-Flash)
- [Technical Blog](https://static.stepfun.com/blog/step-3.7-flash/)
- [StepFun API Platform](https://platform.stepfun.ai)
- [Discord Community](https://discord.gg/RcMJhNVAQc)

**Model Highlights:**
- 198B-parameter sparse Mixture-of-Experts (MoE) model, ~11B active parameters per token
- 288 routed experts (top-8) plus a shared expert; 45 decoder layers
- Mixed full / sliding-window grouped-query attention with a head-wise attention gate
- 256K token context length
- Three reasoning levels (low / medium / high) and tool-calling support
- Apache 2.0 License

> **Note:** Step-3.7-Flash is a vision-language model. AutoDeploy onboards the **text generation path**; the vision encoder is not deployed through this workflow.

**Prerequisites:**
- 8x NVIDIA GPUs with recent drivers (BF16 weights are ~400 GB total) and CUDA 12.x
- Python 3.10+
- TensorRT-LLM ([container](https://catalog.ngc.nvidia.com/orgs/nvidia/teams/tensorrt-llm/containers/release) or pip install)

## Prerequisites & Environment

Set up a containerized environment for TensorRT-LLM by running the following command in a terminal:

```shell
docker run --rm -it --ipc=host --ulimit memlock=-1 --ulimit stack=67108864 --gpus=all -p 8000:8000 nvcr.io/nvidia/tensorrt-llm/release:1.3.0rc1
```

You now have TensorRT-LLM set up!

In [ ]:
# If pip not found
!python -m ensurepip --default-pip

In [ ]:
%pip install torch openai

## Verify GPU

Check that CUDA is available and the GPUs are detected correctly. Step-3.7-Flash requires 8 GPUs for the BF16 weights.

In [ ]:
# Environment check
import sys

import torch

print(f"Python: {sys.version}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Num GPUs: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU[{i}]: {torch.cuda.get_device_name(i)}")

## OpenAI-Compatible Server

Start a local OpenAI-compatible server with TensorRT-LLM via the terminal, within the running docker container.

Ensure that the following commands are executed from the docker terminal.

Start with the Step-3.7-Flash YAML here: `examples/auto_deploy/model_registry/configs/step-3.7-flash.yaml`

### Load the Model

Launch the TensorRT-LLM server with Step-3.7-Flash:

```shell
trtllm-serve "stepfun-ai/Step-3.7-Flash" \
  --host 0.0.0.0 \
  --port 8000 \
  --backend _autodeploy \
  --trust_remote_code \
  --extra_llm_api_options examples/auto_deploy/model_registry/configs/step-3.7-flash.yaml
```

The same standalone config YAML can be validated locally with `build_and_run_ad.py`:

```shell
python examples/auto_deploy/build_and_run_ad.py \
  --model stepfun-ai/Step-3.7-Flash \
  --args.yaml-extra examples/auto_deploy/model_registry/configs/step-3.7-flash.yaml
```

Your server is now running!

## Use the API

Use the OpenAI-compatible client to send requests to the TensorRT-LLM server.

In [ ]:
from openai import OpenAI

# Setup client
BASE_URL = "http://0.0.0.0:8000/v1"
API_KEY = "null"
client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

MODEL_ID = "stepfun-ai/Step-3.7-Flash"

In [ ]:
# Basic chat completion
print("Chat Completion Example")
print("=" * 50)

response = client.chat.completions.create(
    model=MODEL_ID,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is 15% of 85? Show your reasoning."},
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=512,
)

print("Response:")
print(response.choices[0].message.content)

In [ ]:
# Streaming chat completion
print("Streaming response:")
print("=" * 50)

stream = client.chat.completions.create(
    model=MODEL_ID,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What are the first 5 prime numbers?"},
    ],
    temperature=0.7,
    max_tokens=1024,
    stream=True,
)

for chunk in stream:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)

## Evaluation Parameters

For optimal results, use the following parameters based on your task:

**Default Settings (Reasoning Tasks)**
- `temperature`: 1.0
- `top_p`: 0.95
- `max_tokens`: up to 8192 (model supports a 256K context)

**Deterministic Tasks**
- `temperature`: 0
- `max_tokens`: 16384

Step-3.7-Flash exposes three reasoning levels (low / medium / high); see the model card for how to select them via the chat template.

## Additional Resources

- [TensorRT-LLM Documentation](https://nvidia.github.io/TensorRT-LLM/)
- [AutoDeploy Guide](https://nvidia.github.io/TensorRT-LLM/torch/auto_deploy/auto-deploy.html)
- [Step-3.7-Flash on HuggingFace](https://huggingface.co/stepfun-ai/Step-3.7-Flash)
- [Step-3.7-Flash Technical Blog](https://static.stepfun.com/blog/step-3.7-flash/)
- [StepFun Discord Community](https://discord.gg/RcMJhNVAQc)